In [82]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel, VarianceThreshold, SelectPercentile
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from dotenv import load_dotenv
import joblib
from gene_thesaurus import GeneThesaurus


In [83]:
load_dotenv()

X_train_path = os.getenv("X_TRAIN_CSV_PATH")
y_train_path = os.getenv("Y_TRAIN_TSV_PATH")
X_test_path = os.getenv("X_TEST_CSV_PATH")
y_test_path = os.getenv("Y_TEST_TSV_PATH")
output_dir = os.getenv("OUTPUT_DIR")
random_seed = int(os.getenv("RANDOM_SEED"))
n_jobs = int(os.getenv("N_JOBS", -1))

# Print configuration
print(f"X_train_path: {X_train_path}")
print(f"y_train_path: {y_train_path}")
print(f"X_test_path: {X_test_path}")
print(f"y_test_path: {y_test_path}")
print(f"output_dir: {output_dir}")
print(f"random_seed: {random_seed}")
print(f"n_jobs: {n_jobs}")

X_train_path: /home/mariya/Data/pallas/nf-core-gexml-output-train/transpose/transposed.csv
y_train_path: /home/mariya/Data/pallas/nf-core-gexml-output-train/class/subtypes.tsv
X_test_path: /home/mariya/Data/pallas/nf-core-gexml-output-test/transpose/transposed.csv
y_test_path: /home/mariya/Data/pallas/nf-core-gexml-output-test/class/subtypes.tsv
output_dir: /home/mariya/Data/pallas/training
random_seed: 42
n_jobs: 10


In [84]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)
X_train


,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
id,,,,,,,,,,,,,,,,,,,,,
16-002,-0.788379,-1.240057,4.156152,2.259114,4.095816,3.847072,0.347254,2.704738,4.114815,6.484072,...,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057
16-003,-0.877625,-1.240057,3.708616,4.044157,4.499860,2.924013,0.319119,2.385971,4.271631,6.265382,...,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057
16-006,-0.999385,-1.240057,3.961579,3.801073,3.668684,4.019283,-0.723580,2.381367,3.755395,6.661700,...,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057
16-011,4.450546,-1.240057,4.668189,4.444524,5.415203,4.426273,-0.445605,1.790593,4.937077,7.402061,...,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057
16-014,-0.392027,-1.240057,4.189044,4.239813,3.338585,5.297999,0.162078,2.522018,5.095366,6.597946,...,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GE3309,1.971714,-1.240057,2.539159,2.169154,3.640737,7.626621,-1.240057,3.431982,7.443087,6.310414,...,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057
GE8321,-1.240057,-1.240057,3.758737,2.320461,2.096506,3.316420,1.140979,3.544402,3.894794,7.789438,...,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057
GE9336,-1.240057,-1.240057,3.578423,3.837078,3.279873,2.200801,1.655443,4.578119,3.284581,6.232270,...,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057


In [85]:
y_train = pd.read_csv(y_train_path, sep="\t", index_col=0)
y_train

,class
sample,
SJAML031525_D1,"Disease,AML,RUNX1::RUNX1T1"
SJAML032978_D1,"Disease,AML,KMT2A-r"
SJBALL032622_D1,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
SJBALL030267_D1,"Disease,B-ALL,aneuploidy,high hyperdiploid"
SJMLL021_D,"Disease,AML,KMT2A-r"
...,...
SJHBBBN9RBJ,"Disease,B-ALL,aneuploidy,high hyperdiploid"
SJHBBBM4X52,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
SJHBBBM2GV5,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"


In [86]:
# align the indices
X_train = X_train.loc[y_train.index]
X_train

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
sample,,,,,,,,,,,,,,,,,,,,,
SJAML031525_D1,-0.387877,0.333831,4.925510,3.335318,3.506543,5.804824,0.893825,4.274806,3.885236,5.462626,...,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057
SJAML032978_D1,-0.874909,-1.240057,3.788826,4.654983,3.943080,8.001503,0.070076,4.648783,5.271382,6.058625,...,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057
SJBALL032622_D1,-0.952136,-1.240057,4.785538,4.545878,4.757837,1.989928,2.117854,0.906425,3.589755,6.757789,...,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057
SJBALL030267_D1,-0.588313,-1.240057,4.349547,4.140317,4.427608,4.340415,2.912507,2.433989,5.247421,6.506839,...,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057
SJMLL021_D,-1.094880,-1.240057,3.844576,4.241710,4.874338,7.624038,0.048816,5.057565,5.178295,6.506255,...,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SJHBBBN9RBJ,2.332443,-1.240057,4.290793,2.766476,4.431261,1.834022,2.864544,3.003058,4.954285,6.061093,...,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057
SJHBBBM4X52,-0.934757,-1.240057,3.843876,4.485755,3.986318,8.157433,0.742246,4.616352,5.119456,6.110335,...,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057
SJHBBBM2GV5,-1.193094,-1.240057,3.873068,4.125265,3.820114,4.483878,1.405297,2.877571,4.047370,6.744054,...,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057,-1.240057


In [89]:
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, sep="\t", index_col=0)
y_test

,class
sample,
SJBALL046129,"Disease,B-ALL,aneuploidy,high hyperdiploid"
SJBALL043683,"Disease,B-ALL,BCR::ABL1-group,BCR::ABL1"
SJHYPER134,"Disease,B-ALL,aneuploidy,high hyperdiploid"
SJBALL015152,"Disease,B-ALL,BCR::ABL1-group,BCR::ABL1-like"
SJALL015965,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
...,...
GE0310,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
ALLT-358,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
ALLT-325,"Disease,B-ALL,KMT2A-r"


In [90]:
# align the indices
X_test = X_test.loc[y_test.index]
X_test

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
sample,,,,,,,,,,,,,,,,,,,,,
SJBALL046129,-0.121991,-1.228701,4.865514,3.231017,4.227107,3.287877,0.627767,3.505486,5.171740,7.726325,...,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701
SJBALL043683,-1.228701,-1.228701,3.696276,4.500919,4.229144,5.213761,0.157852,3.924989,4.109891,7.825652,...,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701
SJHYPER134,0.237859,-1.228701,4.204345,3.033172,2.937280,4.487640,-0.259088,2.737274,4.387522,5.848464,...,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701
SJBALL015152,0.539324,-1.228701,4.468654,4.431808,4.884632,5.217601,4.591120,4.754862,4.903868,6.466505,...,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701
SJALL015965,-0.525387,-0.758162,4.565420,2.242319,3.470591,3.466634,-0.146531,1.214191,3.056177,5.275060,...,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GE0310,0.092808,-1.228701,4.382275,2.645359,3.190163,3.261369,2.051726,5.333404,4.854624,7.098830,...,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701
ALLT-358,-0.545220,-1.228701,4.732709,4.286579,5.101769,5.966723,0.572319,3.531696,5.248293,6.476796,...,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701
ALLT-325,-0.415973,-1.228701,4.223331,3.838671,4.133059,-0.390237,1.669071,3.384754,3.885034,6.504425,...,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701,-1.228701


In [91]:
# Get least frequently ocurring class count
min_class_count = y_train.value_counts().min()
min_class_count

np.int64(3)

In [92]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array
y_test = y_test.iloc[:, 0].values
y_test

array(['Disease,B-ALL,aneuploidy,high hyperdiploid',
       'Disease,B-ALL,BCR::ABL1-group,BCR::ABL1',
       'Disease,B-ALL,aneuploidy,high hyperdiploid',
       'Disease,B-ALL,BCR::ABL1-group,BCR::ABL1-like',
       'Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1',
       'Disease,B-ALL,KMT2A-r', 'Disease,B-ALL,TCF3::PBX1',
       'Disease,B-ALL,BCR::ABL1-group,BCR::ABL1-like',
       'Disease,B-ALL,BCR::ABL1-group,BCR::ABL1-like',
       'Disease,B-ALL,PAX5alt', 'Disease,B-ALL,BCR::ABL1-group,BCR::ABL1',
       'Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1',
       'Disease,B-ALL,BCR::ABL1-group,BCR::ABL1-like',
       'Disease,B-ALL,BCR::ABL1-group,BCR::ABL1', 'Disease,B-ALL,PAX5alt',
       'Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1',
       'Disease,B-ALL,DUX4-r',
       'Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1-like',
       'Disease,B-ALL,TCF3::PBX1',
       'Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1',
       'Disease,B-ALL,KMT2A-r',
       'Disease,B-ALL,aneuploidy,high hyp

In [ ]:
N_ITER=30
N_REPEATS=1 # WE WILL VALIDATE FINAL PARAMS USING MORE REPEATS

# ----------------------
# Pipeline: Variance filter -> RF feature selection -> RF classifier
# ----------------------
pipeline = Pipeline([
    ("variance_filter", VarianceThreshold(threshold=0.01)), # remove near-constant genes
    ("feature_selection", SelectPercentile(percentile=10)),  # keep only top 10% of features
    ("rf", RandomForestClassifier(random_seed, n_jobs=-n_jobs))
])

# ----------------------
# Hyperparameter search space
# ----------------------
param_grid = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [None, 20, 50],
    'rf__max_features': ['sqrt', 'log2'],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 5],
    'rf__class_weight': [None, 'balanced']
}

# ----------------------
# Cross-validation + randomized search
# ----------------------
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=N_ITER,
    cv=cv,
    scoring="f1_weighted",   # good for multiclass imbalance
    n_jobs=-n_jobs,
    random_state=random_seed,
    verbose=1
)

search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print("Best CV (F1-weighted):", search.best_score_)
best_pipeline = search.best_estimator_

## SAVE EVERYTHING
# Save best hyperparameters
joblib.dump(search.best_params_, os.path.join(output_dir, "best_hyperparameters.pkl"))

# Save features - USE BEST_PIPELINE INSTEAD OF PIPELINE
# First get features that passed the variance threshold
variance_mask = best_pipeline.named_steps['variance_filter'].get_support()
remaining_features = X_train.columns[variance_mask]

# Then get features selected by SelectFromModel
selected_mask = best_pipeline.named_steps['feature_selection'].get_support()
selected_features = remaining_features[selected_mask]

joblib.dump(selected_features, os.path.join(output_dir, "selected_features.pkl"))

# Save the whole pipeline
joblib.dump(best_pipeline, os.path.join(output_dir, "best_pipeline.pkl"))

Fitting 3 folds for each of 30 candidates, totalling 90 fits
Best hyperparameters: {'rf__n_estimators': 200, 'rf__min_samples_split': 10, 'rf__min_samples_leaf': 5, 'rf__max_features': 'sqrt', 'rf__max_depth': 50, 'rf__class_weight': 'balanced'}
Best CV (F1-weighted): 0.9096312906371846


['/home/mariya/Data/pallas/training/best_pipeline.pkl']

In [94]:
# Cross-validate again with more repeats and larger estimators
N_REPEATS=10

# Use just the selected features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# Read in "best_hyperparameters.pkl"
with open(os.path.join(output_dir, "best_hyperparameters.pkl"), "rb") as f:
    best_params = joblib.load(f)

best_params = {k.replace('rf__', ''): v for k, v in best_params.items()}
best_params['n_estimators'] = 500  # Use more trees for final model
print("Final model parameters:", best_params)
final_rf = RandomForestClassifier(**best_params,
                            n_jobs=n_jobs,
                            random_state=random_seed)

# Cross-validate again with more repeats
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

cv_scores = cross_val_score(final_rf,
                           X_train_selected,
                           y_train,
                           cv=cv,
                           scoring="f1_weighted",
                           n_jobs=-n_jobs,
                           verbose=1)

print(f"Cross-validation F1-weighted: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Train the final model on all training data
final_rf.fit(X_train_selected, y_train)

# Save the final model
joblib.dump(final_rf, os.path.join(output_dir, "final_model.pkl"))

Final model parameters: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 50, 'class_weight': 'balanced'}


[Parallel(n_jobs=-10)]: Using backend LokyBackend with 5 concurrent workers.


Cross-validation F1-weighted: 0.909 ± 0.007


['/home/mariya/Data/pallas/training/final_model.pkl']

In [95]:
# ----------------------
# Evaluate final model on test set
# ----------------------

# Use selected features for test set
y_pred_final = final_rf.predict(X_test_selected)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_test_selected.shape}")

print("\nTest set classification report (final optimized model):")
print(classification_report(y_test, y_pred_final, digits=3))

# ----------------------
# Extract top genes from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Gene": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

gt = GeneThesaurus()
translated = gt.translate_genes(gene_importances["Gene"].tolist(),
                               source="ensembl_id", target="symbol")
gene_importances["Gene"] = gene_importances["Gene"].map(translated)

print("\nTop 20 genes driving classification (final model):")
print(gene_importances.head(20))

Number of selected features: 1967
Shape of X_test_selected: (515, 1967)

Test set classification report (final optimized model):
                                                  precision    recall  f1-score   support

                       Disease,AML,APL-PML::RARA      1.000     0.500     0.667         2
                      Disease,AML,CBFA2T3::GLIS2      1.000     1.000     1.000         2
                         Disease,AML,CBFB::MYH11      1.000     1.000     1.000         9
                            Disease,AML,CEBPAalt      1.000     0.800     0.889         5
                         Disease,AML,DEK::NUP214      0.000     0.000     0.000         2
                             Disease,AML,KMT2A-r      0.824     0.875     0.848        16
                             Disease,AML,NPM1alt      0.636     0.875     0.737         8
                             Disease,AML,NUP98-r      1.000     0.714     0.833         7
                      Disease,AML,RUNX1::RUNX1T1      0.882 

/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri


Top 20 genes driving classification (final model):
         Gene  Importance
1626    NUTM1    0.006933
937    MEGF10    0.006284
899     MEIS1    0.004542
1909    APELA    0.003936
1223     EBF1    0.003631
497       PDC    0.003438
283     SALL4    0.003177
1113    AGAP1    0.003135
1088   TSPAN7    0.003088
1130  IGF2BP1    0.003060
697    ATP8A2    0.003034
1554   ADGRD2    0.002924
1722    GREB1    0.002891
136    INPP5A    0.002878
1648     PBX1    0.002871
66      STAP1    0.002866
79      GPM6B    0.002701
1586     TTC3    0.002660
1635   ANKS1B    0.002659
1773   KLHL14    0.002655


In [97]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 0 out of 1967

Bottom 20 genes by importance (final model):
          Gene    Importance
1858     TCL1B  3.210580e-05
1770       SPN  3.173774e-05
585        NaN  3.067740e-05
565   SLC25A16  3.007156e-05
927      CCNA2  3.000601e-05
1429     OXSR1  2.686140e-05
534       ETF1  2.521215e-05
935       BTF3  2.323458e-05
1118       XDH  2.238983e-05
1617       PGP  1.996624e-05
1384     OR4D5  1.942596e-05
1074     CXXC1  1.860924e-05
1708     SF3B3  1.843695e-05
1899       MIF  1.580332e-05
1111      CIB4  1.306920e-05
966      PSKH2  1.160850e-05
1681      SSX7  1.144887e-05
961      FATE1  1.088504e-05
1493     OR4N2  9.306729e-06
1765       HTT  8.837032e-07
